# Memory vs RAG
This notebook is self-sufficient and follows Write -> Manage -> Read.
It supports local or cloud mode via environment variables.

## Copilot SDK Companion (Option B)
A runnable companion snippet is included below.


In [ ]:
import os
from pathlib import Path
import json

RUN_MODE = os.getenv('AGENTIC_RUN_MODE', 'local').lower()
USE_CLOUD = RUN_MODE == 'cloud'
PROJECT_ENDPOINT = os.getenv('FOUNDRY_PROJECT_ENDPOINT', '')
MODEL = os.getenv('FOUNDRY_MODEL', 'gpt-4o')

print({'mode': RUN_MODE, 'cloud': USE_CLOUD, 'model': MODEL})


In [ ]:
data_root = Path('..') / 'data'
sample = {'user':'E001','query':'book Seattle trip'}
employees_path = data_root / 'employees.json'

if employees_path.exists():
    employees = json.loads(employees_path.read_text(encoding='utf-8'))
else:
    employees = [{'employee_id':'E001','name':'Sample User'}]

print('employees', len(employees), 'sample', sample['query'])


In [ ]:
def write_manage_read_demo(prompt: str):
    working = [{'role':'user','text': prompt}]
    managed = working[-3:]
    retrieved = managed[0]['text'] if managed else ''
    return {'write': len(working), 'manage': len(managed), 'read': retrieved}

print(write_manage_read_demo(sample['query']))


In [ ]:
def copilot_sdk_companion_snippet(prompt: str):
    # ponytail: minimal runnable companion, replace with SDK client wiring as needed.
    return {'sdk':'copilot', 'prompt': prompt, 'status':'ready'}

print(copilot_sdk_companion_snippet('Summarize memory state'))


In [ ]:
search_endpoint = os.getenv('AZURE_SEARCH_ENDPOINT', '')
search_index = os.getenv('AZURE_SEARCH_INDEX', 'travel-memory')
search_key = os.getenv('AZURE_SEARCH_ADMIN_KEY', '')
docs = [{'id': '1', 'content': 'Domestic trips default to economy.'}, {'id': '2', 'content': 'User E001 prefers aisle seats.'}]
if USE_CLOUD and search_endpoint and search_key:
    try:
        from azure.core.credentials import AzureKeyCredential
        from azure.search.documents import SearchClient
        from azure.search.documents.indexes import SearchIndexClient
        from azure.search.documents.indexes.models import SearchIndex, SearchableField, SimpleField, SearchFieldDataType
        cred = AzureKeyCredential(search_key)
        fields = [SimpleField(name='id', type=SearchFieldDataType.String, key=True), SearchableField(name='content', type=SearchFieldDataType.String)]
        SearchIndexClient(search_endpoint, cred).create_or_update_index(SearchIndex(name=search_index, fields=fields))
        SearchClient(search_endpoint, search_index, cred).upload_documents(docs)
        print('cloud index seeded', search_index, len(docs))
    except Exception as exc:
        print('cloud seed skipped:', exc)
else:
    print('local docs loaded:', len(docs))